# Klasifikacija galaksija — trening na Colab-u

Pokretati **ćelije redom, odozgo nadole**.

Pre prvog pokretanja podesi GPU: **Runtime → Change runtime type → Hardware accelerator → T4 GPU**.

---
### Šta treba da bude na Google Drive-u

U folderu `MyDrive/diplomski/` treba da stoje dva zip fajla:

- `kod.zip` — folder `src/` plus `app.py` i `requirements.txt`
- `podaci.zip` — pripremljen folder `data/gz_hubble_500/` (bez `_parquet_cache`)

Rezultati se automatski vraćaju na Drive posle svakog modela, pa prekid sesije ne znači gubitak posla.


## 1. Provera GPU-a


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)
print('CUDA dostupna:', torch.cuda.is_available())
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Nema GPU-a! Runtime -> Change runtime type -> T4 GPU'


## 2. Povezivanje Drive-a


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Raspakivanje koda i podataka

Sve se raspakuje na lokalni disk Colab-a, jer je čitanje slika sa Drive-a znatno sporije.


In [ ]:
import os, shutil, zipfile, pathlib

# Folder se trazi bez obzira na velika/mala slova.
MOJ_DRIVE = pathlib.Path('/content/drive/MyDrive')
kandidati = [d for d in MOJ_DRIVE.iterdir() if d.is_dir() and d.name.lower() == 'diplomski']
assert kandidati, f"Nema foldera 'diplomski' u {MOJ_DRIVE}"
DRIVE = kandidati[0]
print('Drive folder:', DRIVE)

PROJEKAT = pathlib.Path('/content/galaksije')
shutil.rmtree(PROJEKAT, ignore_errors=True)          # cist start pri svakom pokretanju
ODREDISTA = {'kod.zip': PROJEKAT, 'podaci.zip': PROJEKAT / 'data'}


def raspakuj(zip_putanja: pathlib.Path, odrediste: pathlib.Path) -> int:
    """Raspakuje zip napravljen na Windows-u.

    PowerShell Compress-Archive upisuje putanje sa obrnutom kosom crtom, pa
    Linux ne vidi foldere nego fajlove sa imenom 'src\\gz_classifier\\train.py'.
    Zato se imena clanova rucno prevode u '/' pre snimanja.
    """
    broj = 0
    with zipfile.ZipFile(zip_putanja) as z:
        for clan in z.infolist():
            ime = clan.filename.replace('\\', '/')
            if ime.endswith('/') or '__pycache__' in ime or not ime.strip():
                continue
            cilj = odrediste / ime
            cilj.parent.mkdir(parents=True, exist_ok=True)
            with z.open(clan) as ulaz, open(cilj, 'wb') as izlaz:
                shutil.copyfileobj(ulaz, izlaz)
            broj += 1
    return broj


for naziv, odrediste in ODREDISTA.items():
    izvor = DRIVE / naziv
    assert izvor.exists(), f'Nedostaje {izvor}. Na Drive-u ima: {sorted(f.name for f in DRIVE.iterdir())}'
    odrediste.mkdir(parents=True, exist_ok=True)
    print(f'raspakovano {raspakuj(izvor, odrediste):5d} fajlova iz {naziv} -> {odrediste}')

os.chdir(PROJEKAT)
print('\nradni folder:', os.getcwd())
print('sadrzaj:', sorted(os.listdir('.')))
assert (PROJEKAT / 'src' / 'gz_classifier' / 'train.py').exists(), 'kod.zip nema src/gz_classifier/'
assert (PROJEKAT / 'data' / 'gz_hubble_500' / 'metadata.csv').exists(), 'podaci.zip nema gz_hubble_500/metadata.csv'
print('putanje ispravne')



## 4. Instalacija biblioteka

Colab već ima PyTorch, NumPy, OpenCV i scikit-learn. Dodaje se samo ono što fali.


In [ ]:
!pip install -q timm datasets transformers
import timm, datasets, transformers
print('timm', timm.__version__, '| datasets', datasets.__version__, '| transformers', transformers.__version__)


## 5. Provera skupa podataka

Ako brojevi po klasama nisu ravnomerni ili nema sva tri splita, stani ovde.


In [ ]:
import pandas as pd

PODACI = 'data/gz_hubble_500'
meta = pd.read_csv(f'{PODACI}/metadata.csv')
pregled = meta.pivot_table(index='label', columns='split', values='path', aggfunc='count').fillna(0).astype(int)
display(pregled)
print('ukupno slika:', len(meta))
assert {'train','val','test'} <= set(meta['split']), 'Nedostaje neki split!'


## 5b. Provera lanca


In [ ]:
# ista provera lanca koja je prosla lokalno
!python proba_pipeline.py


## 6. Kontrolno pokretanje

Jedan model, oba uslova, svega nekoliko epoha — samo da se potvrdi da ceo lanac radi
i da brojevi imaju smisla. Nasumično pogađanje na 5 klasa daje macro-F1 oko **0.20**;
posle par epoha sa pretreniranim težinama očekuj osetno više.


In [ ]:
!python -m src.gz_classifier.run_modality_experiment \
    --data-dir data/gz_hubble_500 \
    --output-root /content/proba \
    --models resnet18 \
    --epochs 3 --patience 3 \
    --overwrite


## 7. Glavni eksperiment

Sedam arhitektura × dva uslova ulaza = 14 treninga.
Posle svakog modela rezultati se kopiraju na Drive.

Ako sesija pukne, samo pokreni ovu ćeliju ponovo — završeni treninzi se preskaču.


In [ ]:
import subprocess, shutil, sys, pathlib, zipfile

MODELI = ['resnet18', 'mobilenet_v3_small', 'efficientnet_b0', 'densenet121',
          'vit_tiny_patch16_224', 'deit_tiny_patch16_224', 'swin_tiny_patch4_window7_224']

IZLAZ_LOKALNO = pathlib.Path('/content/outputs_500_pretrained')
IZLAZ_DRIVE   = DRIVE / 'outputs_500_pretrained'
IZLAZ_DRIVE.mkdir(parents=True, exist_ok=True)

EPOHA, STRPLJENJE = 60, 10


def vrati_sa_drajva():
    """Raspakuje ranije sacuvane runove nazad na lokalni disk."""
    vraceno = []
    for z in sorted(IZLAZ_DRIVE.glob('*.zip')):
        with zipfile.ZipFile(z) as f:
            f.extractall(IZLAZ_LOKALNO)
        vraceno.append(z.stem)
    return vraceno


def sacuvaj_na_drajv(run_dir: pathlib.Path) -> bool:
    """Pakuje jedan run u JEDAN zip i proverava da je stvarno stigao.

    Ranija verzija je koristila shutil.copytree direktno na Drive. To ume da
    zataji bez greske - ispise da je gotovo, a fajlovi ne stignu. Jedan veci
    fajl Drive prenosi pouzdano, a provera velicine to i potvrdjuje.
    """
    privremeni = pathlib.Path('/content') / f'{run_dir.name}.zip'
    with zipfile.ZipFile(privremeni, 'w', zipfile.ZIP_STORED) as z:
        for fajl in sorted(run_dir.iterdir()):
            if fajl.is_file() and fajl.name != 'last.ckpt':
                z.write(fajl, f'{run_dir.name}/{fajl.name}')

    ocekivano = privremeni.stat().st_size
    odrediste = IZLAZ_DRIVE / privremeni.name
    shutil.copy2(privremeni, odrediste)
    privremeni.unlink()

    stiglo = odrediste.stat().st_size if odrediste.exists() else 0
    if stiglo != ocekivano:
        print(f'   !! NIJE PRENET: {odrediste.name} ({stiglo/1e6:.1f} od {ocekivano/1e6:.1f} MB)')
        return False
    print(f'   sacuvano na Drive: {odrediste.name}  {stiglo/1e6:.1f} MB')
    return True


vraceni = vrati_sa_drajva()
if vraceni:
    print('vraceno sa Drive-a:', vraceni, '\n')

neuspeh = []
for redni, model in enumerate(MODELI, 1):
    print(f'\n{"="*70}\n[{redni}/{len(MODELI)}] {model}\n{"="*70}')
    ishod = subprocess.run([
        sys.executable, '-m', 'src.gz_classifier.run_modality_experiment',
        '--data-dir', 'data/gz_hubble_500',
        '--output-root', str(IZLAZ_LOKALNO),
        '--models', model,
        '--epochs', str(EPOHA), '--patience', str(STRPLJENJE),
    ])
    if ishod.returncode != 0:
        print(f'!! {model} nije uspeo, nastavljam dalje')
        neuspeh.append(model)
        continue
    for uslov in ('original', 'multimodal'):
        rd = IZLAZ_LOKALNO / f'{model}_{uslov}'
        if rd.exists() and not sacuvaj_na_drajv(rd):
            neuspeh.append(f'{model}_{uslov}')

print('\n' + '='*70)
print('GOTOVO' if not neuspeh else f'ZAVRSENO UZ PROBLEME: {neuspeh}')



## 7b. Provera da je sve stiglo na Drive


In [ ]:
# OBAVEZNA PROVERA pre nego sto zatvoris sesiju
import zipfile

zipovi = sorted(IZLAZ_DRIVE.glob('*.zip'))
print(f'{"zip na Drive-u":<44} {"MB":>8}   sadrzaj')
print('-' * 88)
ukupno = 0
for z in zipovi:
    mb = z.stat().st_size / 1e6
    ukupno += mb
    with zipfile.ZipFile(z) as f:
        imena = sorted(pathlib.Path(n).name for n in f.namelist())
    ima_ckpt = 'best.ckpt' in imena
    print(f'{z.name:<44} {mb:8.1f}   {"OK" if ima_ckpt else "NEMA best.ckpt!"}  {imena}')

print(f'\nukupno {len(zipovi)} zipova, {ukupno:.0f} MB')
if len(zipovi) < 14:
    print('UPOZORENJE: ocekivano je 14 zipova (7 modela x 2 uslova)')
else:
    print('Sve je na Drive-u. Sada je bezbedno zatvoriti sesiju.')



## 8. Zbirni rezultati


In [ ]:
# Zbirni rezultati
import pandas as pd

zbir = pd.read_csv(IZLAZ_LOKALNO / 'summary.csv')
kolone = [c for c in ['model','condition','best_val_macro_f1','test_macro_f1','test_acc',
                      'delta_test_macro_f1','epochs_run','best_epoch'] if c in zbir.columns]
display(zbir[kolone].sort_values(['model','condition']))

zbir.to_csv(IZLAZ_DRIVE / 'summary.csv', index=False)
print('summary.csv sacuvan na Drive')



## 9. Анализе — GradCAM, аблација, робусност

Покреће се **у истој сесији**, док су тежине још на локалном диску.


In [ ]:
# SVE TRI ANALIZE - u istoj sesiji, dok su tezine jos na lokalnom disku.
#
# Ovo je namerno ovde, a ne kasnije: ako prenos tezina na Drive iz bilo kog
# razloga zataji, slike i brojevi su vec napravljeni i sacuvani.
import subprocess, sys, pathlib

ANALIZE = pathlib.Path('/content/analize')
ANALIZE.mkdir(parents=True, exist_ok=True)
PAROVI = ['densenet121', 'swin_tiny_patch4_window7_224']

zadaci = [
    ('GradCAM slike i doprinos modaliteta po klasama',
     ['slike_gradcam.py', '--output-root', str(IZLAZ_LOKALNO), '--data-dir', 'data/gz_hubble_500',
      '--models', *PAROVI, '--per-class', '2', '--out', str(ANALIZE)]),
    ('Ablacija modaliteta',
     ['ablacija.py', '--output-root', str(IZLAZ_LOKALNO), '--data-dir', 'data/gz_hubble_500',
      '--out', str(ANALIZE)]),
    ('Robusnost na degradaciju slike',
     ['robusnost.py', '--output-root', str(IZLAZ_LOKALNO), '--data-dir', 'data/gz_hubble_500',
      '--models', *PAROVI, '--out', str(ANALIZE)]),
]

neuspele = []
for naslov, komanda in zadaci:
    print(f'\n{"="*70}\n{naslov}\n{"="*70}')
    ishod = subprocess.run([sys.executable] + komanda)
    if ishod.returncode != 0:
        print(f'!! {naslov} nije uspelo')
        neuspele.append(naslov)

print('\n' + '='*70)
print('sve analize gotove' if not neuspele else f'PROBLEMI: {neuspele}')
print('\nnapravljeni fajlovi:')
for f in sorted(ANALIZE.iterdir()):
    print(f'  {f.name:<44} {f.stat().st_size/1e3:8.1f} KB')



## 10. Слање анализа на Drive и завршна провера


In [ ]:
# Analize na Drive - kao jedan zip, uz proveru velicine
import shutil, zipfile

privremeni = pathlib.Path('/content/analize.zip')
with zipfile.ZipFile(privremeni, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(ANALIZE.rglob('*')):
        if f.is_file():
            z.write(f, f.relative_to(ANALIZE))

odrediste = DRIVE / 'analize.zip'
shutil.copy2(privremeni, odrediste)
ok = odrediste.exists() and odrediste.stat().st_size == privremeni.stat().st_size
print(f'analize.zip na Drive: {odrediste.stat().st_size/1e6:.2f} MB  ->  {"OK" if ok else "NIJE PRENET!"}')

# jos jednom i checkpointi, za slucaj da je nesto promaklo
zipovi = sorted(IZLAZ_DRIVE.glob('*.zip'))
print(f'\ncheckpoint zipova na Drive: {len(zipovi)} / 14')
if len(zipovi) < 14:
    print('UPOZORENJE: nedostaju neki. Pokreni celiju 7 ponovo - preskocice zavrsene.')
else:
    print('Sve je na Drive-u. Sesija se moze zatvoriti.')



## 11. Поређење са тренингом од нуле (опционо)

Два модела без претренираних тежина. Ради се само ако остане времена и GPU квоте.


In [ ]:
!python -m src.gz_classifier.run_modality_experiment \
    --data-dir data/gz_hubble_500 \
    --output-root /content/outputs_500_scratch \
    --models densenet121 vit_tiny_patch16_224 \
    --from-scratch \
    --epochs 40 --patience 10

import shutil
shutil.copytree('/content/outputs_500_scratch', DRIVE / 'outputs_500_scratch', dirs_exist_ok=True)
print('sinhronizovano na Drive')
